In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/extracted-josn/latex_extracted.json
/kaggle/input/saved-model-and-dataset/val_dataset/val_dataset/state.json
/kaggle/input/saved-model-and-dataset/val_dataset/val_dataset/dataset_info.json
/kaggle/input/saved-model-and-dataset/val_dataset/val_dataset/data-00000-of-00001.arrow
/kaggle/input/saved-model-and-dataset/bart_summarizes_final/config.json
/kaggle/input/saved-model-and-dataset/bart_summarizes_final/merges.txt
/kaggle/input/saved-model-and-dataset/bart_summarizes_final/vocab.json
/kaggle/input/saved-model-and-dataset/bart_summarizes_final/tokenizer_config.json
/kaggle/input/saved-model-and-dataset/bart_summarizes_final/model.safetensors
/kaggle/input/saved-model-and-dataset/bart_summarizes_final/special_tokens_map.json
/kaggle/input/saved-model-and-dataset/bart_summarizes_final/generation_config.json
/kaggle/input/metrices-csv/metrics_report_valset.csv


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## T5 Summarizer


In [50]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import re
import json
import pandas as pd
from pandas import json_normalize

In [3]:
model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [4]:
def t5_generate(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    out_ids = model.generate(
        inputs["input_ids"],
        num_beams=4,
        max_length=250,
    )

    summary = tokenizer.decode(out_ids[0], skip_special_tokens=True)
    return summary

In [5]:
def clean_summary(text):
    if not text:
        return ""

    text = re.sub(r"\$[^$]*\$", "", text)
    text = re.sub(r"\$\$[^$]*\$\$", "", text)
    text = re.sub(r"[#|&*{}_^\\\/]+", " ", text)
    text = re.sub(r"\\[a-zA-Z]+\*?", "", text)
    text = re.sub(r"\|[^|]*\|", " ", text)
    text = re.sub(r"(?i)section\s*\d+:?", "", text)
    text = re.sub(r"(?i)summary for section\s*\d+:?", "", text)
    text = re.sub(r"[-=]{2,}", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()

    return text

In [6]:
def build_section_chunks(paper):
    chunks = []
    for sec in paper["sections"]:
        chunks.append({
            "title": paper["title"],
            "abstract": paper["abstract"],
            "section_name": sec["name"],
            "section_content": sec["content"]
        })
    return chunks

In [7]:
def format_equation_for_summary(eq):
    eq = eq.replace("\\\\", "\\\\\n")
    return f"```latex\n{eq}\n```"

In [8]:
def pick_top_equations_raw(paper, top_k=5):
    eqs = paper.get("equations", [])
    return eqs[:top_k]

In [9]:
def summarize_abstract(paper):
    prompt = f"""
        ou are summarizing ONLY the abstract of a scientific research paper.

        Write a detailed academic summary that is 5–7 sentences long.
        Your summary MUST include:
        - the main problem or object studied,
        - the motivation or context (if implied),
        - the methods or approach used (e.g., computation, ML, geometry),
        - the key results or theorems,
        - the significance of those results.

        Strict rules:
        - Summarize ONLY the abstract, NOT any section.
        - Use clear academic writing, not bullet points.
        - DO NOT include equations or LaTeX.
        - DO NOT add new claims beyond what is described.
        - DO NOT shorten the meaning; expand it into full sentences.
        - DO NOT copy the abstract wording; rewrite in new sentences.

        Abstract Text:
        {paper['abstract']}

        ---
        Write the abstract summary below (5–7 full sentences):
            """
    return t5_generate(prompt)

In [10]:
def get_section_instruction(section_name):
    name = section_name.lower()

    if "introduction" in name:
        return """
        Summarize the INTRODUCTION by focusing on:
        - the motivation for the research,
        - what problem the authors want to solve,
        - why the problem is important,
        - the general idea of their approach.
        Do NOT include detailed results or experiments.
        """

    if "result" in name:
        return """
        Summarize the RESULTS section by focusing on:
        - the main theorems or mathematical claims,
        - the surjectivity criterion,
        - how the indeterminacy locus I_f determines surjectivity.
        Do NOT talk about motivation or experiments.
        """

    if "experiment" in name or "example" in name:
        return """
        Summarize the EXPERIMENTS/EXAMPLES section by focusing on:
        - the computational or ML methods used,
        - how Python or algorithms generated examples,
        - new explicit maps the authors constructed,
        - empirical observations that support the theory.
        Do NOT describe theorems.
        """

    return "Summarize this section accurately and clearly."

In [11]:
def build_summary_prompt(chunk, top_equations):
    eq_text = "\n\n".join(format_equation_for_summary(eq) for eq in top_equations)
    section_instruction = get_section_instruction(chunk['section_name'])

    prompt = f"""
    You are summarizing a section of a scientific mathematics paper.
    Follow the instructions below to summarize correctly.

    {section_instruction}

    Section Name: {chunk['section_name']}
    Section Content:
    {chunk['section_content']}

    Important Equations:
    {eq_text}

    Write a detailed, clear, human-friendly summary (5–7 sentences).
    """
    return prompt

In [12]:
def explain_equation_t5(eq_latex):
    prompt = (
        "The following is a LaTeX mathematical equation:\n\n"
        "```latex\n"
        f"{eq_latex}\n"
        "```\n\n"
        "Explain the equation clearly:\n"
        "1. Meaning of variables.\n"
        "2. Mathematical interpretation.\n"
        "3. Step-by-step derivation (as far as possible).\n"
        "4. Why this equation appears in the paper.\n"
        "5. What mathematical object/structure it represents.\n"
    )
    return t5_generate(prompt)

In [54]:
def build_final_report_json( abstract_summary, section_chunks,
        section_summaries, top_equations, derivations, paper):


    #  sections
    sections = []
    for chunk, summary in zip(section_chunks, section_summaries):
        sections.append({
            "section_name": chunk["section_name"],
            "section_summary": summary.strip()
        })

    #  important equations (formatted)
    important_equations = []
    for i, eq in enumerate(top_equations, 1):
        important_equations.append({
            "equation_no": f"Equation {i}",
            "equation": eq
        })

    #  equation explanations (formatted)
    equation_explanations = []
    for i, expl in enumerate(derivations, 1):
        equation_explanations.append({
            "equation_no": f"Equation {i}",
            "explanation": expl.strip()
        })

    #  final json object
    final_json = {
        "title": paper["authors"],
        "authors": paper["title"],
        "abstract": abstract_summary.strip(),
        "sections": sections,
        "important_equations": important_equations,
        "equation_explanations": equation_explanations
    }

    return final_json



In [15]:
json_path = "/kaggle/input/extracted-josn/latex_extracted.json"
with open(json_path, "r") as f:
    data = json.load(f)
paper = data[0]

In [16]:
section_chunks = build_section_chunks(paper)

In [17]:
top_equations = pick_top_equations_raw(paper, top_k=5)

In [18]:
abstract_summary = summarize_abstract(paper)

In [19]:
abstract_summary_clean = clean_summary(abstract_summary)
abstract_summary_clean

'We develop an experimental approach, based on some Python programming and Machine Learning, towards the classification of such maps; a couple of new explicit is constructed in this way. We also prove (via pure projective geometry) that a general non-regular cubic endomorphism of is surjective if and only if the set has cardinality at least .'

In [20]:
section_summaries = []
for chunk in section_chunks:
    prompt = build_summary_prompt(chunk, top_equations)
    summary = t5_generate(prompt)
    print(chunk['section_name'])
    print(summary)
    section_summaries.append(clean_summary(summary))

Introduction
In the present note, we concentrate on the behavior of surjective maps in families, aiming to show that for certain rational maps the surjectivity is a . Namely, let us take $X = 2$, so that in projective coordinates any of its rational endomorphisms looks like this: $$ f = [f_0: f_1: f_2]: 2 2, $$ where $f_i$ are mutually coprime homogeneous polynomials of some degree $d$. These polynomials span a plane $$ in the linear system $|_2(d)|$ of all degree $d$ curves on $2$.
Surjectivity results
Theorems on surjectivity of the plane, having cubical components and $#I_f = 9 - 7$, is . Let us resolve indeterminacies of the map $f: 2 2$ via the blow-up of $I_f$; this is possible due to the generality assumption (cf. ): & X [dl]_ [dr]  2 @-->[rr]f && 2 Here $$ is the blow-up of $P_1,,P_9-$, so that the birational transform $_*-1()$ coincides with the linear system $|-K_X|$ for the canonical class $K_X = *(K_2) + E_1 + + E_9 - $, where $E_i := -1(P_i)$ are the exceptional divisors. 

In [21]:
section_summaries

['In the present note, we concentrate on the behavior of surjective maps in families, aiming to show that for certain rational maps the surjectivity is a . Namely, let us take , so that in projective coordinates any of its rational endomorphisms looks like this: f = [f 0: f 1: f 2]: 2 2, where are mutually coprime homogeneous polynomials of some degree . These polynomials span a plane in the linear system of all degree curves on .',
 'Theorems on surjectivity of the plane, having cubical components and , is . Let us resolve indeterminacies of the map via the blow-up of ; this is possible due to the generality assumption (cf. ): X [dl] [dr] 2 @ >[rr]f 2 Here is the blow-up of , so that the birational transform coincides with the linear system for the canonical class , where are the exceptional divisors. It follows that coincides with a projection of the degree smooth surface $X = X =',
 'Here we consider a particular instance of (general cubic) surjective maps with the locus consisting 

In [23]:
derivations = [explain_equation_t5(eq) for eq in top_equations]

In [46]:
final_data = build_final_report_json(
    abstract_summary,
    section_chunks,
    section_summaries,
    top_equations,
    derivations,
    paper
)
print(final_data)

{'title': ['Ilya Karzhemanov'], 'authors': 'Computations and ML for surjective rational maps', 'abstract': 'We develop an experimental approach, based on some Python programming and Machine Learning, towards the classification of such maps; a couple of new explicit $f$ is constructed in this way. We also prove (via pure projective geometry) that a general non-regular cubic endomorphism $f$ of $2$ is surjective if and only if the set $I_f$ has cardinality at least $3$.', 'sections': [{'section_name': 'Introduction', 'section_summary': 'In the present note, we concentrate on the behavior of surjective maps in families, aiming to show that for certain rational maps the surjectivity is a . Namely, let us take , so that in projective coordinates any of its rational endomorphisms looks like this: f = [f 0: f 1: f 2]: 2 2, where are mutually coprime homogeneous polynomials of some degree . These polynomials span a plane in the linear system of all degree curves on .'}, {'section_name': 'Surje

In [51]:
df = pd.json_normalize(final_data)
df

,title,authors,abstract,sections,important_equations,equation_explanations
0,[Ilya Karzhemanov],Computations and ML for surjective rational maps,"We develop an experimental approach, based on ...","[{'section_name': 'Introduction', 'section_sum...","[{'equation_no': 'Equation 1', 'equation': '\l...","[{'equation_no': 'Equation 1', 'explanation': ..."


In [53]:
df.to_csv("/content/drive/MyDrive/structured_summary.csv", index=False)